# 01 — Exploratory data analysis: IBM AML HI-Small

Each section follows **question → code → output → interpretation**. Decisions that come out of a section move into `src/` as functions with tests; this notebook keeps the evidence and the reasoning.

**Public repo rule:** outputs show aggregates and plots only, never raw rows.

## 0. Setup

In [1]:
import pandas as pd

from skeptical_second_reader.data import (
    ACCOUNTS_CSV,
    ID_COLUMNS,
    KEY_SEPARATOR,
    PATTERNS_TXT,
    account_entities,
    account_key,
    account_labels,
    load_transactions,
    read_accounts,
    read_patterns,
)

## 1. Load

**Question:** does the loader keep the data intact? Two traps in the raw file: both account columns are named `Account`, and bank codes carry leading zeros (`010`), which the pandas pyarrow engine silently drops.

The first call reads the CSV (slow) and caches it as parquet in `data/interim/`; later calls read the parquet.

In [2]:
%%time
df = load_transactions()
print(f"rows: {len(df):,}  columns: {df.shape[1]}")

rows: 5,078,345  columns: 11
CPU times: total: 3.02 s
Wall time: 429 ms


In [3]:
df.dtypes

timestamp             datetime64[us]
from_bank                        str
from_account                     str
to_bank                          str
to_account                       str
amount_received              float64
receiving_currency               str
amount_paid                  float64
payment_currency                 str
payment_format                   str
is_laundering                  int64
dtype: object

In [4]:
# Invariants of the loaded data; the notebook stops here if one fails.
assert df.isna().sum().sum() == 0, "missing values"
assert set(df["is_laundering"].unique()) <= {0, 1}, "label is not binary"
for col in ID_COLUMNS:
    assert pd.api.types.is_string_dtype(df[col]), f"{col} is not a string column"
assert df["from_bank"].str.startswith("0").any(), "leading zeros were dropped"

print("time span:", df["timestamp"].min(), "->", df["timestamp"].max())
print(f"laundering rate: {df['is_laundering'].mean():.4%}")

time span: 2022-09-01 00:00:00 -> 2022-09-18 16:18:00
laundering rate: 0.1019%


**Interpretation:** The data loaded correctly: all invariant checks passed (no missing values, binary label, ID columns stored as strings with leading zeros intact). The dataset contains 5,078,345 transactions covering about 18 days (1–18 September 2022). The laundering rate is 0.1019%, roughly 1 in 980 transactions, so the classes are highly imbalanced. A model that labels every transaction as clean would reach about 99.9% accuracy while catching no laundering at all. Accuracy is therefore not informative here; we will use AUC-PR (area under the precision–recall curve) as the main metric.

## 2. Account identity

**Question:** what identifies an account? Each transaction names a sender and a receiver, each as a (bank code, account number) pair. Two checks decide the identity used everywhere downstream (features, split, cases):

1. Would numeric bank codes merge different banks? (Why IDs are strings.)
2. Does the same account number occur at more than one bank? (Whether the number alone is enough.)

In [5]:
# Check 1: bank codes that would collide if read as numbers.
banks = pd.Series(pd.concat([df["from_bank"], df["to_bank"]]).unique())
as_number = banks.str.lstrip("0").replace("", "0")
codes_per_number = banks.groupby(as_number).nunique()

print(f"distinct bank codes: {len(banks):,}")
print(f"codes starting with 0: {banks.str.startswith('0').sum():,}")
print(f"numeric values shared by 2+ codes: {(codes_per_number > 1).sum():,}")

distinct bank codes: 30,528
codes starting with 0: 30,528
numeric values shared by 2+ codes: 58


In [6]:
# Check 2: account numbers that occur at more than one bank.
pairs = pd.concat(
    [
        df[["from_bank", "from_account"]].set_axis(["bank", "account"], axis=1),
        df[["to_bank", "to_account"]].set_axis(["bank", "account"], axis=1),
    ]
).drop_duplicates()
banks_per_number = pairs.groupby("account")["bank"].nunique()
shared = banks_per_number[banks_per_number > 1]

print(f"distinct (bank, account) pairs: {len(pairs):,}")
print(f"distinct account numbers:       {banks_per_number.size:,}")
print(f"numbers at 2+ banks:            {shared.size:,}")
print(f"most banks for one number:      {shared.max() if shared.size else 0}")

distinct (bank, account) pairs: 515,088
distinct account numbers:       515,080
numbers at 2+ banks:            8
most banks for one number:      2


In [7]:
# The account key used downstream: (bank, account number) joined by a separator.
for col in ID_COLUMNS:
    has_separator = df[col].str.contains(KEY_SEPARATOR, regex=False).any()
    assert not has_separator, f"separator in {col}"

df["from_key"] = account_key(df["from_bank"], df["from_account"])
df["to_key"] = account_key(df["to_bank"], df["to_account"])
n_keys = pd.concat([df["from_key"], df["to_key"]]).nunique()

assert n_keys == len(pairs), "key count differs from distinct (bank, account) pairs"
print(f"distinct account keys: {n_keys:,}")

distinct account keys: 515,088


**Interpretation:** There are 515,088 unique accounts in this dataset. Only 8 of the 515,080 account numbers occur at more than one bank, and each of them at most at two banks. For this reason we identify an account not just by its account number but by the (bank code, account number) pair; otherwise, for these 8 numbers, two different accounts would be merged into one. We accept that the effect is small, but the identity should follow from what an account is, not from what this dataset happens to contain, and it costs only a separator character. A unit test checks that an account gets the same key as sender and as receiver, so one account cannot appear under two keys, which would let it fall into both the training and the test set.

## 3. Account label

**Question:** the label is per transaction, but a case is an account. When is an account positive? Options considered: (A) any role in a laundering transaction, (B) sender only, (C) a minimum count or share of laundering transactions, (D) membership in a listed laundering pattern. Chosen: **A**, with the pattern type kept as metadata (not a model feature). The checks below are the evidence for that choice.

In [8]:
# A versus B, and how much laundering a positive account actually has.
labels = account_labels(df)
laundering = df[df["is_laundering"] == 1]
senders = laundering["from_key"].unique()

n_laundering = pd.concat([laundering["from_key"], laundering["to_key"]]).value_counts()
n_all = pd.concat([df["from_key"], df["to_key"]]).value_counts()
share = n_laundering / n_all.reindex(n_laundering.index)

assert labels.sum() == n_laundering.size, "label count differs from accounts seen"
n_pos = labels.sum()
print(f"accounts: {labels.size:,}  positive (A): {n_pos:,} ({labels.mean():.2%})")
print(f"sender only (B): {len(senders):,}  lost by B: {n_pos - len(senders):,}")
print(f"positives with exactly one laundering tx: {(n_laundering == 1).mean():.1%}")
print(f"laundering share of a positive account's tx, median: {share.median():.1%}")
median_tx = n_all[n_laundering.index].median()
print(f"total tx of a positive account, median: {median_tx:.0f}")

accounts: 515,088  positive (A): 6,357 (1.23%)
sender only (B): 3,376  lost by B: 2,981
positives with exactly one laundering tx: 80.7%
laundering share of a positive account's tx, median: 5.6%
total tx of a positive account, median: 22


In [9]:
# D: how much of the laundering the pattern file covers.
patterns = read_patterns(PATTERNS_TXT)
pattern_keys = pd.Index(
    pd.concat(
        [
            account_key(patterns["from_bank"], patterns["from_account"]),
            account_key(patterns["to_bank"], patterns["to_account"]),
        ]
    ).unique()
)
positives = labels.index[labels == 1]

print(f"attempts: {patterns['attempt'].nunique()}  pattern tx: {len(patterns):,}")
print(f"laundering tx covered by patterns: {len(patterns) / len(laundering):.1%}")
n_inside = pattern_keys.isin(positives).sum()
print(f"pattern accounts: {len(pattern_keys):,}  inside A: {n_inside:,}")
print(patterns.groupby("pattern")["attempt"].nunique().to_string())

attempts: 370  pattern tx: 3,209
laundering tx covered by patterns: 62.0%
pattern accounts: 3,170  inside A: 3,170
pattern
BIPARTITE         49
CYCLE             54
FAN-IN            40
FAN-OUT           48
GATHER-SCATTER    51
RANDOM            41
SCATTER-GATHER    44
STACK             43


In [10]:
# Laundering by day: normal activity ends earlier than the laundering patterns.
day = df["timestamp"].dt.date
by_day = df.groupby(day)["is_laundering"].agg(transactions="size", laundering="sum")
by_day["laundering_rate"] = (by_day["laundering"] / by_day["transactions"]).round(4)
by_day

,transactions,laundering,laundering_rate
timestamp,,,
2022-09-01,1114921,322,0.0003
2022-09-02,754449,408,0.0005
2022-09-03,207382,391,0.0019
2022-09-04,207430,407,0.0020
2022-09-05,482650,471,0.0010
2022-09-06,482089,531,0.0011
2022-09-07,482751,497,0.0010
2022-09-08,482773,539,0.0011
2022-09-09,654467,514,0.0008


In [11]:
# The tail after the last day of normal activity (the paper reports 10 days).
last_normal_day = pd.Timestamp("2022-09-10").date()
tail = by_day[by_day.index > last_normal_day]
normal = by_day[by_day.index <= last_normal_day]
n_tail_laundering = tail["laundering"].sum()

assert n_tail_laundering + normal["laundering"].sum() == len(laundering)
print(f"tail days: {len(tail)}  max transactions per day: {tail['transactions'].max()}")
rate_min, rate_max = tail["laundering_rate"].min(), tail["laundering_rate"].max()
print(f"tail laundering rate: {rate_min:.1%} to {rate_max:.1%}")
rate_min, rate_max = normal["laundering_rate"].min(), normal["laundering_rate"].max()
print(f"1-10 Sep laundering rate: {rate_min:.2%} to {rate_max:.2%}")
share_tail = n_tail_laundering / len(laundering)
print(f"laundering tx in tail: {n_tail_laundering} ({share_tail:.1%} of all)")
print(f"laundering tx 1-10 Sep: {normal['laundering'].sum():,}")

tail days: 8  max transactions per day: 396
tail laundering rate: 56.5% to 72.7%
1-10 Sep laundering rate: 0.03% to 0.21%
laundering tx in tail: 655 (12.7% of all)
laundering tx 1-10 Sep: 4,522


**Interpretation:** Each account is labelled as positive if it took part in at least one laundering transaction, as sender or as receiver: 6,357 of the 515,088 accounts. If we labelled only senders, we would lose 2,981 of these accounts. These receivers may include accounts that collect laundered money without belonging to the launderer, for example employees or suppliers paid by a criminal with illicit funds (Altman et al., 2023, p. 6).

The label follows the source of the money, and it is transitive: when illicit funds move along a chain of accounts, every transaction in the chain is labelled as laundering (Altman et al., 2023, p. 7). The listed patterns cover only 62% of the laundering transactions; the rest is laundering outside the patterns (Altman et al., 2023, p. 6). Because of that, a positive account is an account that sent or received illicit funds, not necessarily an account controlled by a launderer. This likely explains why 80.7% of the positive accounts have exactly one laundering transaction.

According to the dataset paper, all accounts used in a laundering pattern belong to the laundering entity or are under its control (Altman et al., 2023, p. 4); the pattern file lists 3,170 such accounts. We do not give the pattern type (for example FAN-OUT or CYCLE) to the model; we keep it only as metadata, to select demo cases and to stratify the sample of the LLM experiment. A bank does not know in advance which pattern an account belongs to; this is known only after an investigation. In this dataset, the pattern type would also leak the label: all 3,170 pattern accounts are positive.

The dataset covers 18 days, but after 10 September the number of transactions drops to at most 396 per day, and 56.5–72.7% of these transactions are laundering. Between 1 and 10 September the laundering rate is only 0.03–0.21%. The dataset paper reports 10 days for HI-Small (Altman et al., 2023, Table 4), so the days after 10 September look like a tail after the normal activity ends. This tail contains 655 laundering transactions, 12.7% of all laundering. If this tail leaks into the features, the model can learn the calendar instead of laundering; because of that, we will decide the time window of the features in the feature step. The paper also reports 3.6K laundering transactions for HI-Small, but we count 5,177 in the file, while the laundering rate (1 in 981) matches. We could not explain this difference; all numbers in this notebook are measured from the dataset file.

## 4. Entities and the split unit

**Question:** what must stay on one side of the train/test split? The dataset paper says some entities control more than one account (Altman et al., 2023, p. 4). If accounts of the same entity land in both train and test, the model may recognise the owner rather than the behaviour, and test scores are inflated. Options: (A) account-disjoint split, (B) entity-disjoint split (all accounts of an entity on one side). Three checks decide between them:

1. Does the accounts file join to the transaction accounts? (Bank code format, coverage.)
2. How many entities have more than one account, and how large is the largest?
3. Do multi-account entities mix positive and negative accounts? (This is where an account split can leak.)

In [12]:
# Check 1: the accounts file, read with string IDs (same leading-zero trap).
accounts = read_accounts(ACCOUNTS_CSV)

tx_banks = pd.Series(pd.concat([df["from_bank"], df["to_bank"]]).unique())
file_banks = accounts["bank"].drop_duplicates()
tx_zero = tx_banks.str.startswith("0").mean()
file_zero = file_banks.str.startswith("0").mean()
print(
    f"bank codes with a leading zero: transactions {tx_zero:.0%}, file {file_zero:.0%}"
)
direct = labels.index.isin(account_key(accounts["bank"], accounts["account"]))
print(f"direct join: {direct.sum():,} of {direct.size:,} transaction accounts found")

bank codes with a leading zero: transactions 100%, file 0%


direct join: 0 of 515,088 transaction accounts found


In [13]:
# The file drops the leading zeros, so join on the unpadded bank code.
parts = labels.index.to_series().str.split(KEY_SEPARATOR, n=1, expand=True)
join_key = account_key(parts[0].str.lstrip("0").replace("", "0"), parts[1])
file_key = account_key(accounts["bank"], accounts["account"])
assert join_key.is_unique, "two transaction accounts merge without leading zeros"
assert file_key.is_unique, "duplicate accounts in the accounts file"

found = join_key.isin(file_key)
print(f"unpadded join: {found.sum():,} of {found.size:,} found ({found.mean():.2%})")
n_extra = (~file_key.isin(join_key)).sum()
print(f"accounts in the file but not in transactions: {n_extra:,}")

unpadded join: 515,088 of 515,088 found (100.00%)


accounts in the file but not in transactions: 3,493


In [14]:
# Check 2: accounts per entity, for the accounts that appear in the transactions.
# Same unpadded join, through the tested function in src/ (raises on any gap).
entities = account_entities(labels.index, read_accounts(ACCOUNTS_CSV))
acc = labels.to_frame().assign(entity_id=entities)

per_entity = acc.groupby("entity_id").size()
multi_ids = per_entity.index[per_entity > 1]
in_multi = acc["entity_id"].isin(multi_ids)
print(f"entities: {per_entity.size:,}  with 2+ accounts: {multi_ids.size:,}")
print(f"accounts in multi-account entities: {in_multi.sum():,} ({in_multi.mean():.2%})")
print(f"largest entity: {per_entity.max():,} accounts")
print(per_entity.value_counts().sort_index().head(10).to_string())

entities: 166,207  with 2+ accounts: 63,236
accounts in multi-account entities: 412,117 (80.01%)
largest entity: 7,820 accounts
1     102971
2      15979
3      11610
4       9827
5       8370
6       6059
7       4134
8       2412
9       1365
10       753


In [15]:
# Check 3: label mix inside multi-account entities (where an account split leaks).
by_entity = acc.groupby("entity_id")["label"].agg(n="size", pos="sum")
multi = by_entity[by_entity["n"] > 1]
mixed = multi[(multi["pos"] > 0) & (multi["pos"] < multi["n"])]
all_pos = multi[multi["pos"] == multi["n"]]

n_pos = labels.sum()
pos_in_multi = multi["pos"].sum()
print(f"multi-account entities with a positive: {(multi['pos'] > 0).sum():,}")
print(f"  mixed (positive and negative): {len(mixed):,}")
print(f"  all positive: {len(all_pos):,}")
print(f"positives in multi-account entities: {pos_in_multi:,} of {n_pos:,}")
print(f"share of all positives: {pos_in_multi / n_pos:.1%}")

multi-account entities with a positive: 3,396
  mixed (positive and negative): 3,394
  all positive: 2
positives in multi-account entities: 4,272 of 6,357
share of all positives: 67.2%


In [16]:
# Check 4: do positives cluster inside entities more than chance would give?
# Expected count if positives were spread at random over accounts (base rate r).
r = labels.mean()
sharing = multi.loc[multi["pos"] >= 2, "pos"].sum()
expected = (multi["n"] * r * (1 - (1 - r) ** (multi["n"] - 1))).sum()
print(f"positives sharing an entity with another positive: {sharing:,}")
print(f"expected under random placement: {expected:,.0f}")

big = by_entity[by_entity["n"] > 1000]
big_share = big["n"].sum() / len(acc)
big_rate = big["pos"].sum() / big["n"].sum()
print(f"entities with >1,000 accounts: {len(big)} ({big_share:.1%} of accounts)")
print(f"positive rate in them: {big_rate:.2%}  overall: {r:.2%}")

positives sharing an entity with another positive: 1,148
expected under random placement: 1,604
entities with >1,000 accounts: 22 (12.0% of accounts)
positive rate in them: 0.79%  overall: 1.23%


In [17]:
# Who are the owners? The entity name carries the owner type ("Corporation #1").
owner_type = accounts.drop_duplicates("entity_id")["entity_name"]
print(owner_type.str.replace(r" #\d+$", "", regex=True).value_counts().to_string())

largest = per_entity.idxmax()
largest_keys = acc.index[acc["entity_id"] == largest]
n_banks = largest_keys.str.split(KEY_SEPARATOR).str[0].nunique()
n_largest_pos = by_entity.loc[largest, "pos"]
print(f"largest owner: {per_entity.max():,} accounts at {n_banks:,} banks")
print(f"  share of accounts: {per_entity.max() / len(acc):.1%}")
print(f"  positives: {n_largest_pos} ({n_largest_pos / n_pos:.1%} of all positives)")

entity_name
Corporation            55530
Sole Proprietorship    55369
Partnership            55170
Individual               134
Country                    2
Direct                     2
largest owner: 7,820 accounts at 1,185 banks
  share of accounts: 1.5%
  positives: 49 (0.8% of all positives)


**Interpretation:** The split will keep all accounts of an entity on the same side (entity-disjoint). An entity is the owner of an account; in this dataset almost all owners are businesses (corporations, sole proprietorships, partnerships), and one owner can hold accounts at many banks. The accounts file gives the owner of each account, but it writes bank codes without leading zeros, while all transaction bank codes have them; a direct join finds none of the 515,088 accounts. After removing the zeros, all accounts are found and no two accounts merge.

63,236 of the 166,207 owners have two or more accounts, and together they hold 80% of all accounts. The largest owner alone has 7,820 accounts at 1,185 banks (1.5% of all accounts), with 49 positives (0.8% of all positives). 67.2% of the positive accounts belong to owners with several accounts, spread over 3,396 owners, and in 3,394 of them the owner also has negative accounts. With an account-disjoint split, accounts of the same owner could be in both train and test. If a model can recognise an owner from the features, it can predict the label of that owner's other accounts without learning the behaviour (Saeb et al., 2017, p. 2); test scores would then be optimistic.

The positives do not cluster inside owners: 1,148 positives share an owner with another positive, fewer than the 1,604 expected if positives were spread at random. So an account-disjoint split would probably be only slightly optimistic. The literature calls this case nonindependence between train and test samples (Kapoor & Narayanan, 2022, p. 5), but whether it is leakage depends on the question (Little et al., 2017, p. 4): it is not leakage if the deployed system sees the same dependence structure (Kapoor & Narayanan, 2022, p. 5). A bank that screens its existing customers does see the same owners again. We still choose the entity-disjoint split as the more cautious test (it measures performance on owners the model has never seen), because it costs little (166,207 groups, the largest about 1.5% of the accounts). Comparing both splits can be a robustness check later.

## References

Altman, E., Blanuša, J., von Niederäusern, L., Egressy, B., Anghel, A., Atasu, K. (2023). Realistic synthetic financial transactions for anti-money laundering models. *Advances in Neural Information Processing Systems 36 (NeurIPS 2023), Datasets and Benchmarks Track*.

Kapoor, S., Narayanan, A. (2022). Leakage and the reproducibility crisis in ML-based science. *arXiv:2207.07048*. https://arxiv.org/abs/2207.07048

Little, M. A., Varoquaux, G., Saeb, S., Lonini, L., Jayaraman, A., Mohr, D. C., Kording, K. P. (2017). Using and understanding cross-validation strategies. Perspectives on Saeb et al. *GigaScience*, 6(5), 1–6. https://doi.org/10.1093/gigascience/gix020

Saeb, S., Lonini, L., Jayaraman, A., Mohr, D. C., Kording, K. P. (2017). The need to approximate the use-case in clinical machine learning. *GigaScience*, 6(5), 1–9. https://doi.org/10.1093/gigascience/gix019